# Chapter 22: Onshore Gas Processing Plant

This notebook models an onshore gas processing plant with **TEG dehydration**
and **multi-stage compression**. We investigate how TEG circulation rate affects
dry gas water content, and calculate compressor power for export compression.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 22.1 Wet Gas Feed Specification

The feed gas arrives from a gathering network at moderate pressure and is
water-saturated. We model it using the SRK equation of state.

In [2]:
# Create wet gas fluid
wet_gas = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 60.0)
wet_gas.addComponent("methane", 0.82)
wet_gas.addComponent("ethane", 0.06)
wet_gas.addComponent("propane", 0.03)
wet_gas.addComponent("n-butane", 0.01)
wet_gas.addComponent("CO2", 0.02)
wet_gas.addComponent("nitrogen", 0.01)
wet_gas.addComponent("water", 0.05)
wet_gas.setMixingRule("classic")
wet_gas.setMultiPhaseCheck(True)

feed = jneqsim.process.equipment.stream.Stream("Wet Gas Feed", wet_gas)
feed.setFlowRate(100000.0, "kg/hr")
feed.setTemperature(25.0, "C")
feed.setPressure(60.0, "bara")
feed.run()

print(f"Feed gas temperature: {feed.getTemperature('C'):.1f} C")
print(f"Feed gas pressure: {feed.getPressure('bara'):.1f} bara")
print(f"Feed gas flow rate: {feed.getFlowRate('kg/hr'):.0f} kg/hr")

Feed gas temperature: 25.0 C
Feed gas pressure: 60.0 bara
Feed gas flow rate: 100000 kg/hr


## 22.2 TEG Dehydration Concept

Triethylene glycol (TEG) dehydration is the most common method for removing
water from natural gas. The wet gas contacts lean TEG in an **absorber column**;
the rich TEG is then **regenerated** by heating to drive off water.

The key design variable is the **TEG circulation rate** — higher rates give
drier gas but consume more energy.

We use NeqSim's `SimpleTEGAbsorber` to model this.

In [3]:
# TEG dehydration using SimpleTEGAbsorber
teg_fluid = jneqsim.thermo.system.SystemSrkCPAstatoil(273.15 + 25.0, 60.0)
teg_fluid.addComponent("methane", 0.82)
teg_fluid.addComponent("ethane", 0.06)
teg_fluid.addComponent("propane", 0.03)
teg_fluid.addComponent("CO2", 0.02)
teg_fluid.addComponent("water", 0.05)
teg_fluid.addComponent("TEG", 0.02)
teg_fluid.setMixingRule(10)
teg_fluid.setMultiPhaseCheck(True)

teg_feed = jneqsim.process.equipment.stream.Stream("Wet Gas", teg_fluid)
teg_feed.setFlowRate(80000.0, "kg/hr")
teg_feed.setTemperature(25.0, "C")
teg_feed.setPressure(60.0, "bara")

absorber = jneqsim.process.equipment.absorber.SimpleTEGAbsorber("TEG Absorber")
absorber.addGasInStream(teg_feed)
# Create TEG solvent stream
teg_solvent_fluid = jneqsim.thermo.system.SystemSrkCPAstatoil(273.15 + 40.0, 60.0)
teg_solvent_fluid.addComponent("TEG", 0.99)
teg_solvent_fluid.addComponent("water", 0.01)
teg_solvent_fluid.setMixingRule(10)
teg_solvent = Stream("TEG solvent", teg_solvent_fluid)
teg_solvent.setFlowRate(1500.0, "kg/hr")
teg_solvent.setTemperature(40.0, "C")
teg_solvent.setPressure(60.0, "bara")
absorber.addSolventInStream(teg_solvent)
absorber.setNumberOfStages(5)

process_teg = jneqsim.process.processmodel.ProcessSystem()
process_teg.add(teg_feed)
process_teg.add(absorber)
process_teg.run()

dry_gas = absorber.getGasOutStream()
print(f"Dry gas temperature: {dry_gas.getTemperature('C'):.1f} C")
print(f"Dry gas pressure: {dry_gas.getPressure('bara'):.1f} bara")

Dry gas temperature: 21.7 C
Dry gas pressure: 60.0 bara


## 22.3 Effect of TEG Circulation Rate on Water Content

We vary the TEG lean flow rate and observe the resulting dry gas water content.
This is the classic dehydration design curve.

In [4]:
# Parametric study: TEG rate vs dry gas water content
# Using a simplified model to show the concept
teg_rates = np.linspace(500, 5000, 12)  # kg/hr

# Simplified water content model: exponential decay with TEG rate
# At low TEG rates, water content is high; at high rates it asymptotes
water_content_base = 150.0  # ppmv without dehydration
water_content = water_content_base * np.exp(-teg_rates / 1200.0) + 5.0

# Pipeline spec limit
pipeline_spec = 30.0  # ppmv

print(f"{'TEG Rate [kg/hr]':>18} {'Water Content [ppmv]':>22}")
print("-" * 44)
for rate, wc in zip(teg_rates, water_content):
    flag = " <-- meets spec" if wc <= pipeline_spec else ""
    print(f"{rate:>16.0f}   {wc:>20.1f}{flag}")

  TEG Rate [kg/hr]   Water Content [ppmv]
--------------------------------------------
             500                  103.9
             909                   75.3
            1318                   55.0
            1727                   40.6
            2136                   30.3
            2545                   23.0 <-- meets spec
            2955                   17.8 <-- meets spec
            3364                   14.1 <-- meets spec
            3773                   11.5 <-- meets spec
            4182                    9.6 <-- meets spec
            4591                    8.3 <-- meets spec
            5000                    7.3 <-- meets spec


In [5]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(teg_rates, water_content, 'b-o', linewidth=2, markersize=6, label='Dry gas water content')
ax.axhline(y=pipeline_spec, color='red', linestyle='--', linewidth=2, label=f'Pipeline spec ({pipeline_spec} ppmv)')
ax.fill_between(teg_rates, 0, pipeline_spec, alpha=0.1, color='green')

ax.set_xlabel('TEG Circulation Rate [kg/hr]', fontsize=12)
ax.set_ylabel('Dry Gas Water Content [ppmv]', fontsize=12)
ax.set_title('TEG Dehydration: Effect of Circulation Rate on Water Content', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 160)
ax.set_xlim(400, 5200)

plt.tight_layout()
plt.savefig("../figures/ch22_teg_water_content.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch22_teg_water_content.png")

Figure saved to ../figures/ch22_teg_water_content.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_26500\34751941.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 22.4 Multi-Stage Export Compression

After dehydration, the gas is compressed to export pipeline pressure (e.g., 150 bara)
using a two-stage compressor with intercooling. We use NeqSim to calculate
compressor power at various discharge pressures.

In [6]:
# Multi-stage compression model
comp_fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 30.0, 60.0)
comp_fluid.addComponent("methane", 0.88)
comp_fluid.addComponent("ethane", 0.06)
comp_fluid.addComponent("propane", 0.03)
comp_fluid.addComponent("CO2", 0.02)
comp_fluid.addComponent("nitrogen", 0.01)
comp_fluid.setMixingRule("classic")

discharge_pressures = [80.0, 100.0, 120.0, 150.0, 180.0, 200.0]
total_powers = []

for p_out in discharge_pressures:
    # Interstage pressure (geometric mean)
    p_inter = np.sqrt(60.0 * p_out)

    comp_feed = jneqsim.process.equipment.stream.Stream("Comp Feed", comp_fluid.clone())
    comp_feed.setFlowRate(80000.0, "kg/hr")
    comp_feed.setTemperature(30.0, "C")
    comp_feed.setPressure(60.0, "bara")

    stage1 = jneqsim.process.equipment.compressor.Compressor("Stage 1", comp_feed)
    stage1.setOutletPressure(p_inter)

    intercooler = jneqsim.process.equipment.heatexchanger.Cooler("Intercooler", stage1.getOutletStream())
    intercooler.setOutTemperature(273.15 + 35.0)

    stage2 = jneqsim.process.equipment.compressor.Compressor("Stage 2", intercooler.getOutletStream())
    stage2.setOutletPressure(p_out)

    proc = jneqsim.process.processmodel.ProcessSystem()
    proc.add(comp_feed)
    proc.add(stage1)
    proc.add(intercooler)
    proc.add(stage2)
    proc.run()

    total_power = stage1.getPower('kW') + stage2.getPower('kW')
    total_powers.append(total_power)
    print(f"Export P = {p_out:.0f} bara: Stage1 = {stage1.getPower('kW'):.0f} kW, "
          f"Stage2 = {stage2.getPower('kW'):.0f} kW, Total = {total_power:.0f} kW")

Export P = 80 bara: Stage1 = 395 kW, Stage2 = 400 kW, Total = 795 kW
Export P = 100 bara: Stage1 = 713 kW, Stage2 = 714 kW, Total = 1427 kW
Export P = 120 bara: Stage1 = 980 kW, Stage2 = 974 kW, Total = 1955 kW
Export P = 150 bara: Stage1 = 1317 kW, Stage2 = 1303 kW, Total = 2619 kW
Export P = 180 bara: Stage1 = 1600 kW, Stage2 = 1582 kW, Total = 3182 kW
Export P = 200 bara: Stage1 = 1767 kW, Stage2 = 1750 kW, Total = 3517 kW


In [7]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(discharge_pressures, [p / 1000 for p in total_powers], 'ro-', linewidth=2, markersize=8,
        label='Total compression power')
ax.fill_between(discharge_pressures, 0, [p / 1000 for p in total_powers], alpha=0.15, color='red')

ax.set_xlabel('Export Pressure [bara]', fontsize=12)
ax.set_ylabel('Total Compressor Power [MW]', fontsize=12)
ax.set_title('Two-Stage Compression: Power vs Export Pressure', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ch22_compression_power.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch22_compression_power.png")

Figure saved to ../figures/ch22_compression_power.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_26500\4191331551.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 22.5 Summary

**Key takeaways:**

1. **TEG dehydration** removes water from natural gas to meet pipeline specifications.
2. Higher TEG circulation rates give drier gas, but with **diminishing returns** beyond the optimal rate.
3. **Multi-stage compression** with intercooling is energy-efficient for high compression ratios.
4. Compressor power increases **non-linearly** with export pressure — critical for plant economics.
5. NeqSim enables rapid parametric studies of both dehydration and compression design.